## Data Loading

In [115]:
%%time

import kpex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
from openpyxl import Workbook

# create once, pass around multiple times for writing metrics
wb = Workbook()

warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
file_path = r"..\input_data\results-filtered-dataset.txt"


feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms


CPU times: total: 0 ns
Wall time: 1.82 ms


In [116]:
# Loading and preprocessing data
cat_df= kpex_dev.load_excel_data(file_path, timeout_replacement)
kpex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kpex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# Assigning weights
kpex_dev.assign_weights(cat_df, scheme, epsilon)

# kpex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kpex_dev.post_process_data(cat_df)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,...,UMKP-S,UMKP-R,UMKP-P,UMKP+-S,UMKP+-R,UMKP+-P,kPEX,kPEX-S,kPEX-R,kPEX-P
0,12month1,20,52328,1585220,30.294,9214,57,19,39,16785.102564,...,1800.000000,1800.000000,1800.000000,131.0240,121.2490,117.7020,116.0158,124.7390,125.8084,126.1804
1,130bit,8,584,6059,10.375,322,12,8,5,564.400000,...,3.196156,1.179364,0.653300,2.3643,0.7011,0.8350,1.4934,0.4922,0.9836,1.7554
2,130bit,9,584,6059,10.375,322,12,8,5,564.400000,...,39.330518,2.159862,2.485482,34.7253,3.2734,3.0159,38.6679,4.9357,22.5385,47.5120
3,130bit,10,583,6051,10.379,321,12,9,4,559.500000,...,9.695851,7.271823,1.946958,9.4931,3.0985,2.4184,10.8117,2.0049,10.4852,12.4310
4,144,4,144319,1072349,7.430,26,9,7,3,140266.333333,...,0.588537,0.539013,0.755349,1.0679,1.0694,0.9626,0.9761,1.0700,1.0626,1.1065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4351,zhishi-zhwiki-internallink,7,4666,248342,53.224,2014,99,19,81,2293.913580,...,2.349277,2.272406,2.238443,1.8562,2.2227,2.2188,1.8133,1.8370,2.2205,2.1942
4352,zhishi-zhwiki-internallink,8,6109,313420,51.305,2421,101,19,83,2731.265060,...,3.851166,3.097895,2.829056,2.8021,2.4912,2.5297,2.8437,2.3888,2.3317,2.8890
4353,zhishi-zhwiki-internallink,9,5981,309664,51.775,2406,101,20,82,2688.695122,...,6.984801,2.970734,2.855774,2.4142,2.4424,2.7942,2.4207,2.5012,2.5222,2.9436
4354,zhishi-zhwiki-internallink,10,4419,244006,55.217,1989,99,22,78,2204.871795,...,13.381666,7.638818,3.442978,1.8692,1.8513,1.6981,1.7289,1.7092,1.7741,1.7088


# Training with Random Forest



## Scheme 1

Each algorithm acts as an independent class

Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [117]:
from data_splits import prepare_train_test_strat_Y

X_train, X_test, y_train, y_test, weights_train, weights_test = prepare_train_test_strat_Y(
    cat_df,
    tie_tolerance = 0.0  # algorithms within tie_tolerance% of min treated as ties
)

Expanded 7 rows with near ties using tolerance 0.0%.


In [118]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=4,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)



RandomForestClassifier(class_weight='balanced', max_depth=20,
                       min_samples_split=4, n_estimators=200)

Feature Importances

In [119]:

# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_strat_y.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
0         k_value    0.125666
8   shell_avgSize    0.090445
3           d_avg    0.088160
10  shell_minSize    0.087126
2               m    0.082255
4           d_max    0.081988
7       shell_num    0.079184
6        core_min    0.076047
5        core_max    0.075339
9   shell_maxSize    0.074029
1               n    0.073979
11  shell_stdSize    0.065782


#### Inference on 20% dataset

In [120]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=rf_strat_y,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # 
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/RF_results/tie_tol/0/expand_and_split/exp01',
    file_suffix= "inference_test_set_without_holdout_strat_y_exp_02" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/RF_results/tie_tol/0/expand_and_split/model_performance_for_all_exp.xlsx", 
                                    'Inference_test_set_strat_y_exp_02') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 47.54%

Precision by Algorithm:
  UMKP-S: 47.40% (154 predictions)
  UMKP-R: 44.68% (94 predictions)
  UMKP-P: 62.50% (232 predictions)
  UMKP+-S: 48.28% (58 predictions)
  UMKP+-R: 20.69% (29 predictions)
  UMKP+-P: 30.61% (49 predictions)
  kPEX: 55.40% (139 predictions)
  kPEX-S: 37.21% (43 predictions)
  kPEX-R: 7.69% (26 predictions)
  kPEX-P: 22.45% (49 predictions)

Recall by Algorithm:
  UMKP-S: 51.05% (73/143 cases)
  UMKP-R: 40.38% (42/104 cases)
  UMKP-P: 65.61% (145/221 cases)
  UMKP+-S: 41.18% (28/68 cases)
  UMKP+-R: 18.75% (6/32 cases)
  UMKP+-P: 28.57% (16/56 cases)
  kPEX: 63.71% (79/124 cases)
  kPEX-S: 32.00% (16/50 cases)
  kPEX-R: 8.00% (2/25 cases)
  kPEX-P: 20.75% (11/53 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 59988.58
  Total Optimal Time   = 21823.43
  Time Ratio (pred/opt)= 2.749
  Avg Overhead/case ((pred - opt) / opt)    = 4547.38%

#### Re-evalute on entire dataset

In [121]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=rf_strat_y,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/RF_results/tie_tol/0/expand_and_split/exp01',
    file_suffix= "inference_entire_data_without_holdout_strat_y_exp_02" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/RF_results/tie_tol/0/expand_and_split/model_performance_for_all_exp.xlsx", 
                                    'Inference_entire_data_strat_y_exp_02') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 81.24%

Precision by Algorithm:
  UMKP-S: 79.67% (723 predictions)
  UMKP-R: 77.67% (533 predictions)
  UMKP-P: 87.37% (1053 predictions)
  UMKP+-S: 84.06% (320 predictions)
  UMKP+-R: 70.00% (180 predictions)
  UMKP+-P: 78.75% (273 predictions)
  kPEX: 86.51% (593 predictions)
  kPEX-S: 79.69% (256 predictions)
  kPEX-R: 63.69% (157 predictions)
  kPEX-P: 75.37% (268 predictions)

Recall by Algorithm:
  UMKP-S: 80.45% (576/716 cases)
  UMKP-R: 79.62% (414/520 cases)
  UMKP-P: 83.33% (920/1104 cases)
  UMKP+-S: 80.12% (270/337 cases)
  UMKP+-R: 80.38% (127/158 cases)
  UMKP+-P: 77.70% (216/278 cases)
  kPEX: 84.48% (517/612 cases)
  kPEX-S: 81.27% (204/251 cases)
  kPEX-R: 80.00% (100/125 cases)
  kPEX-P: 77.10% (202/262 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 158774.30
  Total Optimal Time   = 111146.05
  Time Ratio (pred/opt)= 1.429
  Avg Overhead/case ((pr

### **Balanced train-test split based on both Y and k**

In [122]:
from data_splits import prepare_train_test_strat_Y_K

X_train, X_test, y_train, y_test, weights_train, weights_test = prepare_train_test_strat_Y_K(
    cat_df,
    tie_tolerance = 0.00  # algorithms within tie_tolerance% of min treated as ties
)

Expanded 7 rows with near ties using tolerance 0.0%.


In [123]:
# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

RandomForestClassifier(class_weight='balanced', max_depth=20, n_estimators=400)

Feature Importances

In [142]:
# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_strat_y_K.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
0         k_value    0.151824
3           d_avg    0.087850
2               m    0.081957
6        core_min    0.081632
7       shell_num    0.080699
8   shell_avgSize    0.079882
4           d_max    0.077086
10  shell_minSize    0.076236
5        core_max    0.076025
9   shell_maxSize    0.070275
1               n    0.068394
11  shell_stdSize    0.068140


#### Inference on 20% dataset

In [125]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=rf_strat_y_K,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # 
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/RF_results/tie_tol/0/expand_and_split/exp01',
    file_suffix= "inference_test_set_without_holdout_strat_y_K_exp_02" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/RF_results/tie_tol/0/expand_and_split/model_performance_for_all_exp.xlsx", 
                                    'Inference_test_set_strat_y_K_exp_02') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 47.31%

Precision by Algorithm:
  UMKP-S: 55.32% (141 predictions)
  UMKP-R: 39.29% (112 predictions)
  UMKP-P: 59.59% (245 predictions)
  UMKP+-S: 40.62% (64 predictions)
  UMKP+-R: 43.48% (23 predictions)
  UMKP+-P: 25.58% (43 predictions)
  kPEX: 46.98% (149 predictions)
  kPEX-S: 36.36% (44 predictions)
  kPEX-R: 15.00% (20 predictions)
  kPEX-P: 28.12% (32 predictions)

Recall by Algorithm:
  UMKP-S: 54.55% (78/143 cases)
  UMKP-R: 41.90% (44/105 cases)
  UMKP-P: 65.77% (146/222 cases)
  UMKP+-S: 38.24% (26/68 cases)
  UMKP+-R: 29.41% (10/34 cases)
  UMKP+-P: 24.07% (13/54 cases)
  kPEX: 56.45% (70/124 cases)
  kPEX-S: 34.00% (17/50 cases)
  kPEX-R: 12.50% (3/24 cases)
  kPEX-P: 17.31% (9/52 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 48891.45
  Total Optimal Time   = 17300.53
  Time Ratio (pred/opt)= 2.826
  Avg Overhead/case ((pred - opt) / opt)    = 429.4

#### Re-evalute on entire dataset

In [143]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=rf_strat_y_K,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # 
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/RF_results/tie_tol/0/expand_and_split/exp01',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K_exp_02" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/RF_results/tie_tol/0/expand_and_split/model_performance_for_all_exp.xlsx", 
                                    'Inference_entire_data_strat_y_K_exp_02') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 85.49%

Precision by Algorithm:
  UMKP-S: 88.09% (697 predictions)
  UMKP-R: 82.56% (539 predictions)
  UMKP-P: 89.10% (1119 predictions)
  UMKP+-S: 86.15% (325 predictions)
  UMKP+-R: 80.49% (164 predictions)
  UMKP+-P: 80.74% (270 predictions)
  kPEX: 85.90% (603 predictions)
  kPEX-S: 83.67% (251 predictions)
  kPEX-R: 76.30% (135 predictions)
  kPEX-P: 81.82% (253 predictions)

Recall by Algorithm:
  UMKP-S: 85.75% (614/716 cases)
  UMKP-R: 85.58% (445/520 cases)
  UMKP-P: 90.31% (997/1104 cases)
  UMKP+-S: 83.38% (281/337 cases)
  UMKP+-R: 83.54% (132/158 cases)
  UMKP+-P: 79.14% (220/278 cases)
  kPEX: 85.13% (521/612 cases)
  kPEX-S: 84.06% (211/251 cases)
  kPEX-R: 82.40% (103/125 cases)
  kPEX-P: 79.01% (207/262 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 147880.90
  Total Optimal Time   = 111146.05
  Time Ratio (pred/opt)= 1.331
  Avg Overhead/case ((pr

### **Balanced train-test split based on Y**

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

def prepare_train_test_strat_Y_hier(cat_df,
                       weight_col='weight',
                       test_size=0.2,
                       random_state=42,
                       for_nn=False,
                       kcore_list=None,
                       tie_tolerance=0.00):

    if kcore_list:
        feature_cols = ['k_value', 'n', 'm', 'd_avg', 'd_max', 'core_max',
                        'core_min', 'shell_num', 'shell_avgSize',
                        'shell_maxSize', 'shell_minSize', 'shell_stdSize', 'kcore_list']
    elif for_nn:
        feature_cols = ['k_value', 'n', 'm', 'd_avg', 'd_max', 'core_max',
                        'core_min', 'shell_num', 'shell_avgSize',
                        'shell_maxSize', 'shell_minSize', 'shell_stdSize', 'fastest_time']
    else:
        feature_cols = ['k_value', 'n', 'm', 'd_avg', 'd_max', 'core_max',
                        'core_min', 'shell_num', 'shell_avgSize',
                        'shell_maxSize', 'shell_minSize', 'shell_stdSize']

    expanded_rows = []
    num_tied_expansions = 0

    for _, row in cat_df.iterrows():
        values = [row[alg] for alg in algorithms]
        min_val = min(values)

        tied_algs = [
            alg for alg in algorithms
            if row[alg] <= min_val * (1 + tie_tolerance)
        ]

        for alg in tied_algs:
            new_row = row[feature_cols].copy()
            new_row['target'] = alg
            new_row['group'] = ALGO_TO_GROUP[alg]
            new_row['subgroup'] = ALGO_TO_SUBGROUP[alg]
            new_row['weight'] = row[weight_col] / len(tied_algs)
            expanded_rows.append(new_row)

        if len(tied_algs) > 1:
            num_tied_expansions += 1

    expanded_df = pd.DataFrame(expanded_rows)

    print(f"Expanded {num_tied_expansions} rows with near ties using tolerance {tie_tolerance*100:.1f}%.")

    X = expanded_df.drop(['target', 'group', 'subgroup', 'weight'], axis=1)

    y_algo = expanded_df['target']
    y_group = expanded_df['group']
    y_subgroup = expanded_df['subgroup']
    weights = expanded_df['weight']

    X_train, X_test, \
    y_algo_train, y_algo_test, \
    weights_train, weights_test = train_test_split(
        X, y_algo, weights,
        test_size=test_size,
        random_state=random_state,
        stratify=y_algo
    )

    # Align group & subgroup splits using indices
    y_group_train = y_group.loc[X_train.index]
    y_group_test  = y_group.loc[X_test.index]

    y_subgroup_train = y_subgroup.loc[X_train.index]
    y_subgroup_test  = y_subgroup.loc[X_test.index]

    return (X_train, X_test,
            y_algo_train, y_algo_test,
            y_group_train, y_group_test,
            y_subgroup_train, y_subgroup_test,
            weights_train, weights_test)


In [ ]:
from data_splits import prepare_train_test_strat_Y

X_train, X_test, y_train, y_test, weights_train, weights_test = prepare_train_test_strat_Y(
    cat_df,
    tie_tolerance = 0.0  # algorithms within tie_tolerance% of min treated as ties
)

Expanded 7 rows with near ties using tolerance 0.0%.


In [144]:
from sklearn.ensemble import RandomForestClassifier

# ----- Stage 1 -----
group_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# ----- Stage 2 -----
subgroup_clf_factory = lambda: RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# ----- Stage 3 -----
algo_clf_factory = lambda: RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)


In [ ]:
import numpy as np

class HierarchicalClassifier:
    def __init__(
        self,
        group_clf,
        subgroup_clf_factory,
        algo_clf_factory,
        tau_group=0.9,
        tau_subgroup=0.7,
        tau_algo=0.6,
        fallback_label=None
    ):
        self.group_clf = group_clf
        self.subgroup_clf_factory = subgroup_clf_factory
        self.algo_clf_factory = algo_clf_factory

        self.tau_group = tau_group
        self.tau_subgroup = tau_subgroup
        self.tau_algo = tau_algo

        self.fallback_label = fallback_label

        self.subgroup_clfs = {}
        self.algo_clfs = {}

    # --------------------------------------------------
    def fit(self, X, y_group, y_subgroup, y_algo):
        # Stage 1
        self.group_clf.fit(X, y_group)

        # Stage 2
        for g in np.unique(y_group):
            idx = (y_group == g)
            clf = self.subgroup_clf_factory()
            clf.fit(X[idx], y_subgroup[idx])
            self.subgroup_clfs[g] = clf

        # Stage 3
        for sg in np.unique(y_subgroup):
            idx = (y_subgroup == sg)
            clf = self.algo_clf_factory()
            clf.fit(X[idx], y_algo[idx])
            self.algo_clfs[sg] = clf

        self.classes_ = np.unique(y_algo)
        return self
    
    def _confident(self, probs, tau):
        idx = np.argmax(probs)
        return probs[idx] >= tau, idx
    
    def predict(self, X):
        preds = []

        for x in X:
            x = np.asarray(x).reshape(1, -1)

            # ----- Stage 1 -----
            p_group = self.group_clf.predict_proba(x)[0]
            ok, idx = self._confident(p_group, self.tau_group)
            if not ok:
                preds.append(self.fallback_label)
                continue

            group = self.group_clf.classes_[idx]

            # ----- Stage 2 -----
            p_sub = self.subgroup_clfs[group].predict_proba(x)[0]
            ok, idx = self._confident(p_sub, self.tau_subgroup)
            if not ok:
                preds.append(self.fallback_label)
                continue

            subgroup = self.subgroup_clfs[group].classes_[idx]

            # ----- Stage 3 -----
            p_alg = self.algo_clfs[subgroup].predict_proba(x)[0]
            ok, idx = self._confident(p_alg, self.tau_algo)
            if not ok:
                preds.append(self.fallback_label)
                continue

            preds.append(self.algo_clfs[subgroup].classes_[idx])

        return np.array(preds)
    
    def predict_proba(self, X):
        proba = np.zeros((len(X), len(self.classes_)))

        class_to_idx = {c: i for i, c in enumerate(self.classes_)}

        for i, x in enumerate(X):
            x = np.asarray(x).reshape(1, -1)

            p_group = self.group_clf.predict_proba(x)[0]
            group = self.group_clf.classes_[np.argmax(p_group)]

            p_sub = self.subgroup_clfs[group].predict_proba(x)[0]
            subgroup = self.subgroup_clfs[group].classes_[np.argmax(p_sub)]

            p_alg = self.algo_clfs[subgroup].predict_proba(x)[0]
            algs = self.algo_clfs[subgroup].classes_

            for a, p in zip(algs, p_alg):
                proba[i, class_to_idx[a]] = p

        return proba





Feature Importances

In [ ]:

# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_strat_y.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
0         k_value    0.125666
8   shell_avgSize    0.090445
3           d_avg    0.088160
10  shell_minSize    0.087126
2               m    0.082255
4           d_max    0.081988
7       shell_num    0.079184
6        core_min    0.076047
5        core_max    0.075339
9   shell_maxSize    0.074029
1               n    0.073979
11  shell_stdSize    0.065782


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=rf_strat_y,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # 
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/RF_results/tie_tol/0/expand_and_split/exp01',
    file_suffix= "inference_test_set_without_holdout_strat_y_exp_02" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/RF_results/tie_tol/0/expand_and_split/model_performance_for_all_exp.xlsx", 
                                    'Inference_test_set_strat_y_exp_02') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 47.54%

Precision by Algorithm:
  UMKP-S: 47.40% (154 predictions)
  UMKP-R: 44.68% (94 predictions)
  UMKP-P: 62.50% (232 predictions)
  UMKP+-S: 48.28% (58 predictions)
  UMKP+-R: 20.69% (29 predictions)
  UMKP+-P: 30.61% (49 predictions)
  kPEX: 55.40% (139 predictions)
  kPEX-S: 37.21% (43 predictions)
  kPEX-R: 7.69% (26 predictions)
  kPEX-P: 22.45% (49 predictions)

Recall by Algorithm:
  UMKP-S: 51.05% (73/143 cases)
  UMKP-R: 40.38% (42/104 cases)
  UMKP-P: 65.61% (145/221 cases)
  UMKP+-S: 41.18% (28/68 cases)
  UMKP+-R: 18.75% (6/32 cases)
  UMKP+-P: 28.57% (16/56 cases)
  kPEX: 63.71% (79/124 cases)
  kPEX-S: 32.00% (16/50 cases)
  kPEX-R: 8.00% (2/25 cases)
  kPEX-P: 20.75% (11/53 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 59988.58
  Total Optimal Time   = 21823.43
  Time Ratio (pred/opt)= 2.749
  Avg Overhead/case ((pred - opt) / opt)    = 4547.38%

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=rf_strat_y,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/RF_results/tie_tol/0/expand_and_split/exp01',
    file_suffix= "inference_entire_data_without_holdout_strat_y_exp_02" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/RF_results/tie_tol/0/expand_and_split/model_performance_for_all_exp.xlsx", 
                                    'Inference_entire_data_strat_y_exp_02') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 81.24%

Precision by Algorithm:
  UMKP-S: 79.67% (723 predictions)
  UMKP-R: 77.67% (533 predictions)
  UMKP-P: 87.37% (1053 predictions)
  UMKP+-S: 84.06% (320 predictions)
  UMKP+-R: 70.00% (180 predictions)
  UMKP+-P: 78.75% (273 predictions)
  kPEX: 86.51% (593 predictions)
  kPEX-S: 79.69% (256 predictions)
  kPEX-R: 63.69% (157 predictions)
  kPEX-P: 75.37% (268 predictions)

Recall by Algorithm:
  UMKP-S: 80.45% (576/716 cases)
  UMKP-R: 79.62% (414/520 cases)
  UMKP-P: 83.33% (920/1104 cases)
  UMKP+-S: 80.12% (270/337 cases)
  UMKP+-R: 80.38% (127/158 cases)
  UMKP+-P: 77.70% (216/278 cases)
  kPEX: 84.48% (517/612 cases)
  kPEX-S: 81.27% (204/251 cases)
  kPEX-R: 80.00% (100/125 cases)
  kPEX-P: 77.10% (202/262 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 158774.30
  Total Optimal Time   = 111146.05
  Time Ratio (pred/opt)= 1.429
  Avg Overhead/case ((pr

### **Balanced train-test split based on both Y and k**

In [ ]:
from data_splits import prepare_train_test_strat_Y_K

X_train, X_test, y_train, y_test, weights_train, weights_test = prepare_train_test_strat_Y_K(
    cat_df,
    tie_tolerance = 0.00  # algorithms within tie_tolerance% of min treated as ties
)

Expanded 7 rows with near ties using tolerance 0.0%.


In [ ]:
# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

RandomForestClassifier(class_weight='balanced', max_depth=20, n_estimators=400)

Feature Importances

In [ ]:
# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_strat_y_K.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
0         k_value    0.151824
3           d_avg    0.087850
2               m    0.081957
6        core_min    0.081632
7       shell_num    0.080699
8   shell_avgSize    0.079882
4           d_max    0.077086
10  shell_minSize    0.076236
5        core_max    0.076025
9   shell_maxSize    0.070275
1               n    0.068394
11  shell_stdSize    0.068140


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=rf_strat_y_K,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # 
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/RF_results/tie_tol/0/expand_and_split/exp01',
    file_suffix= "inference_test_set_without_holdout_strat_y_K_exp_02" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/RF_results/tie_tol/0/expand_and_split/model_performance_for_all_exp.xlsx", 
                                    'Inference_test_set_strat_y_K_exp_02') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 47.31%

Precision by Algorithm:
  UMKP-S: 55.32% (141 predictions)
  UMKP-R: 39.29% (112 predictions)
  UMKP-P: 59.59% (245 predictions)
  UMKP+-S: 40.62% (64 predictions)
  UMKP+-R: 43.48% (23 predictions)
  UMKP+-P: 25.58% (43 predictions)
  kPEX: 46.98% (149 predictions)
  kPEX-S: 36.36% (44 predictions)
  kPEX-R: 15.00% (20 predictions)
  kPEX-P: 28.12% (32 predictions)

Recall by Algorithm:
  UMKP-S: 54.55% (78/143 cases)
  UMKP-R: 41.90% (44/105 cases)
  UMKP-P: 65.77% (146/222 cases)
  UMKP+-S: 38.24% (26/68 cases)
  UMKP+-R: 29.41% (10/34 cases)
  UMKP+-P: 24.07% (13/54 cases)
  kPEX: 56.45% (70/124 cases)
  kPEX-S: 34.00% (17/50 cases)
  kPEX-R: 12.50% (3/24 cases)
  kPEX-P: 17.31% (9/52 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 48891.45
  Total Optimal Time   = 17300.53
  Time Ratio (pred/opt)= 2.826
  Avg Overhead/case ((pred - opt) / opt)    = 429.4

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=rf_strat_y_K,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # 
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/RF_results/tie_tol/0/expand_and_split/exp01',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K_exp_02" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/RF_results/tie_tol/0/expand_and_split/model_performance_for_all_exp.xlsx", 
                                    'Inference_entire_data_strat_y_K_exp_02') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 85.49%

Precision by Algorithm:
  UMKP-S: 88.09% (697 predictions)
  UMKP-R: 82.56% (539 predictions)
  UMKP-P: 89.10% (1119 predictions)
  UMKP+-S: 86.15% (325 predictions)
  UMKP+-R: 80.49% (164 predictions)
  UMKP+-P: 80.74% (270 predictions)
  kPEX: 85.90% (603 predictions)
  kPEX-S: 83.67% (251 predictions)
  kPEX-R: 76.30% (135 predictions)
  kPEX-P: 81.82% (253 predictions)

Recall by Algorithm:
  UMKP-S: 85.75% (614/716 cases)
  UMKP-R: 85.58% (445/520 cases)
  UMKP-P: 90.31% (997/1104 cases)
  UMKP+-S: 83.38% (281/337 cases)
  UMKP+-R: 83.54% (132/158 cases)
  UMKP+-P: 79.14% (220/278 cases)
  kPEX: 85.13% (521/612 cases)
  kPEX-S: 84.06% (211/251 cases)
  kPEX-R: 82.40% (103/125 cases)
  kPEX-P: 79.01% (207/262 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 147880.90
  Total Optimal Time   = 111146.05
  Time Ratio (pred/opt)= 1.331
  Avg Overhead/case ((pr